# Master and Expanded Data Frame for Machine Learning

This stage involves 2 components:
- Master data frame combining all csv files with pandas
- Expand data by using imbalanced-learn SMOTE analysis

Both components will be done in this Colab Notebook - **04_PPA_df_SMOTE.ipynb**

## Components of Master Data Frame

**Library used to make a df**
- `pandas` -v. 2.2.2

**WHY**: Pandas can handle, manage and create tabular 2D data

**List of all csv files merged to make master data frame:**
- **Dataset 1: Depaul**
    * ppa_depaul_linguistic_features.csv
    * depaul_acoustic_features_parselmouth.csv
    * depaul_acoustic_features_librosa.csv
- **Dataset 2: Hopkins**
    * ppa_hopkins_linguistic_features.csv
    * hopkins_acoustic_features_parselmouth.csv
    * hopkins_acoustic_features_librosa.csv
- **Dataset 3: Baycrest**
    * ppa_baycrest_linguistic_features.csv
    * baycrest_acoustic_features_parselmouth.csv
    * baycrest_acoustic_features_librosa.csv
- **Dataset 4: Pitt**
    * control_pitt_linguistic_features.csv
    * pitt_acoustic_features_parselmouth.csv
    * pitt_acoustic_features_librosa.csv

All CSV files will be joined to make a master data table used for training the machine learning model.

In [ ]:
# 1.Import libraries
import os
import pandas as pd

# 3. Connect Colab to Drive
from google.colab import drive
drive.mount('/content/drive')

# 3. File Paths for all 12 CSV files and Master CSV file
# ling is for linguistic
# pm is for parselmouth acoustic
# lib is for librosa acoustic
dataset_paths = {
    'depaul': {
        'ling': '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/02_PPA_linguistic_extraction_csv/ppa_depaul_linguistic_features.csv',
        'pm':   '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/depaul/depaul_csv/depaul_acoustic_features_parselmouth.csv',
        'lib':  '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/depaul/depaul_csv/depaul_acoustic_features_librosa.csv',
    },
    'hopkins': {
        'ling': '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/02_PPA_linguistic_extraction_csv/ppa_hopkins_linguistic_features.csv',
        'pm':   '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/hopkins/hopkins_csv/hopkins_acoustic_features_parselmouth.csv',
        'lib':  '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/hopkins/hopkins_csv/hopkins_acoustic_features_librosa.csv'
    },
    'baycrest': {
        'ling': '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/02_PPA_linguistic_extraction_csv/ppa_baycrest_linguistic_features.csv',
        'pm':   '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/baycrest/baycrest_csv/baycrest_acoustic_features_parselmouth.csv',
        'lib':  '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/baycrest/baycrest_csv/baycrest_acoustic_features_librosa.csv'
    },
    'pitt': {
        'ling': '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/02_PPA_linguistic_extraction_csv/control_pitt_linguistic_features.csv',
        'pm':   '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/pitt/pitt_csv/pitt_acoustic_features_parselmouth.csv',
        'lib':  '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/pitt/pitt_csv/pitt_acoustic_features_librosa.csv'
    }
}

output_master_csv = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/master_ppa_data_frame.csv'

# 4. Merge Datasets
combined_datasets = [] # Creates an empty list where completed data frames can be stacked together

for dataset_name, paths in dataset_paths.items(): # Loops through the dataset files one by one
    print(f"Merging features for: {dataset_name}...")

    # Reads the three specific CSV files and converts them to a data frame
    df_ling = pd.read_csv(paths['ling'])
    df_pm   = pd.read_csv(paths['pm'])
    df_lib  = pd.read_csv(paths['lib'])

    # Tell Python all patient_id is same as 'participant_id'
    id_translator = {
        'patient_id': 'participant_id',
    }

    # Apply the change to the DataFrames in memory
    df_ling = df_ling.rename(columns=id_translator)
    df_pm   = df_pm.rename(columns=id_translator)
    df_lib  = df_lib.rename(columns=id_translator)

    # Coverts participant_id into str so it can matched across data files without an error
    for df in [df_ling, df_pm, df_lib]:
        if 'participant_id' in df.columns:
            df['participant_id'] = df['participant_id'].astype(str)

    # Removes file_name column when merging to avoid having duplicate columns
    df_pm  = df_pm.drop(columns=['file_name'], errors='ignore')
    df_lib = df_lib.drop(columns=['file_name'], errors='ignore')

    # Horizontal merge using participant_id and outer ensures that missing values get filled with NaN instead of deleting the patient's whole row
    df_merged = df_ling.merge(df_pm, on='participant_id', how='outer')
    df_merged = df_merged.merge(df_lib, on='participant_id', how='outer')

    # Adds new column specifying which dataset the row came from
    df_merged['dataset_source'] = dataset_name
    combined_datasets.append(df_merged)

# 5. Vertically stacks all files into the master data frame where ignore_index=True makes sure row numbers are organized
master_df = pd.concat(combined_datasets, ignore_index=True)

# Puts participant ID and dataset source to the front (far left)
front_cols = ['participant_id', 'dataset_source']
other_cols = [c for c in master_df.columns if c not in front_cols] # Retrieves all every single column name in CSV files
master_df = master_df[front_cols + other_cols] # Merges front + other columns into one

print(f"\n Master DataFrame Built Successfully!") # States if building df is done
print(f"Total Participants: {len(master_df)}") # No. of rows - need to have 305
print(f"Total Features Extracted: {len(master_df.columns) - 2}") # No. of columns without patient ID and dataset source - need to find 62

#6. Save to Drive
os.makedirs(os.path.dirname(output_master_csv), exist_ok=True) # Ensure directory exists
master_df.to_csv(output_master_csv, index=False) # Export

print(f" MASTER CSV SAVED SUCCESSFULLY TO DRIVE!")
print(f"File Path: {output_master_csv}")

## Before SMOTE analysis
**Library used**
- `pandas` -v. 2.2.2

**WHY**: Pandas can handle, manage and create tabular 2D data

**CHANGE 1: Adding patient diagnosis column to original CSV**

**CHANGE 2: Removing rows 25,42,48 (missing acoustic data), and rows with diagnosis of PPA (generic) and unknown**


In [ ]:
# Adding patient diagnosis column to CSV
# 1. Import required libraries
import os
import pandas as pd
from google.colab import drive

# 2. Connect Colab and Drive
drive.mount('/content/drive')

# 3. Load existing CSV from Drive
input_path = "/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/master_ppa_data_frame.csv"
df = pd.read_csv(input_path)

# 4. Add diagnoses with corresponding participant_id
# KEY: svPPA = Semantic ; nfvPPA - nonfluent ; lvPPA - logopenic ; PPA - generic PPA ; unknown - Missing
diagnosis_mapping = {
    "patient_id" : "diagnosis",
    "patient_id_2" : "diagnosis_2"
}
# Data was added in this format to manually add diagnosis to df
# Removed to protect patient privacy and adhere to Talkbank rules

# 5. Tell Python that patient_id is same as participant_id
id_translator = {
    'patient_id': 'participant_id',
}

# 6. Map the diagnoses based on 'participant_id' column
# if no diagnosis is entered for a patient ID, it defaults to control
df["diagnosis"] = df["participant_id"].map(diagnosis_mapping).fillna("control")

# 7. Puts participant ID, diagnosis, and dataset source to the front (far left)
front_cols = ['participant_id', 'diagnosis', 'dataset_source']

# Retrieves every other column name in df
other_cols = [c for c in df.columns if c not in front_cols]

# Reorders the columns in df
df = df[front_cols + other_cols]

# 8. Save as CSV file and Export directly to Drive
output_path = "/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/ppa_master_data_frame_updated.csv"

# Ensure the output folder exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save the updated dataframe
df.to_csv(output_path, index=False)

# Helpful message stating the no. of diagnoses added
print("Updated diagnosis counts:")
print(df["diagnosis"].value_counts())
print(f"\nFile successfully saved to Drive:\n{output_path}")

In [ ]:
# Remove participant data that lacks acoustic data, and if diagnosis is PPA (generic) or unknown
# 1. Import libraries
import pandas as pd
import os

# 2. Connect Colab and Drive
from google.colab import drive
drive.mount('/content/drive')

# 3. Load CSV from drive
# Attach file path
input_path = "/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/ppa_master_data_frame_updated.csv"
df = pd.read_csv(input_path)

# 4. Drop rows 25,42,48 to remove individuals without acoustic data
# errors='ignore' makes sure code doesn't crash if an index doesn't exist
df_filtered = df.drop(index=[25, 42, 48], errors='ignore').copy()

# 5. Remove any row where diagnosis is 'unknown' or 'PPA'
excluded_labels = ['unknown', 'PPA']
df_filtered = df_filtered[~df_filtered['diagnosis'].isin(excluded_labels)].copy()

# 6. Save CSV to Drive
output_path = "/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/ppa_master_data_frame_model.csv"

# Ensures output directory exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Export as CSV
df_filtered.to_csv(output_path, index=False)

## Overview of SMOTE analysis

**Libraries used**
- `imbalanced-learn` -v.0.14.2
- `numpy` - v.2.0.2

**WHY**: Control data outweighs PPA data, resulting in uneven proportions which can create bias when training model.

**IMPORTANT**: Only applied to TRAINING Dataset

**AFTER CHANGES TO ORIGINAL CSV FILE**
Current CSV file ONLY has:
- participant_id with both linguistic and acoustic data
- participant_id with diagnosis either svPPA, nfvPPA, lvPPA or control (excluded generic PPA and unknown diagnoses)

**WHY:** To ensure the project stays focused on **multi-modal approach** and **PPA Variant Classification**.


In [ ]:
# 1. Install and import libraries
!pip install scikit-learn==1.6.0 # Machine learning library
import os # Handles file paths and drive folders
import pandas as pd # Data frames
from sklearn.model_selection import train_test_split # Splits data into training and testing
from imblearn.over_sampling import SMOTE # Expands training data to prevent model training bias

# 2. Connect Drive and Colab
from google.colab import drive
drive.mount('/content/drive')

# 3. Load CSV file with the 2 changes
# Establish File Path
input_path = "/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Results/csv_files/ppa_master_data_frame_model.csv"
# Load CSV file as df
df = pd.read_csv(input_path)
non_feature_cols = ['participant_id', 'dataset_source', 'diagnosis'] # List of informational columns
numeric_cols = [c for c in df.columns if c not in non_feature_cols] # Only takes numeric columns

# 4. Separate Features (X) and Diagnosis (y)
X = df[numeric_cols]
y = df['diagnosis']

# 5. Perform 70/30 Train/Test Split FIRST (Strict No-Leakage Policy)
# 70% - training ; 30% - testing
# random_state = 42 ensures it's reproducible
# stratify=y keeps the % of controls and variants same in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Impute missing NaN values using ONLY training set group avwrages to prevent data leakage
for col in numeric_cols: # Goes through each numeric column
    # 1. Mean for each group using ONLY the training data
    train_group_means = X_train.groupby(y_train)[col].mean() # Creates a table

    # 2. Fill missing values in X_train using its own group means
    X_train[col] = X_train[col].fillna(X_train.groupby(y_train)[col].transform('mean')) # Fills NaNs with matching group average

    # 3. Fill missing values in X_test using the same group averages learned from X_train
    test_group_mean_series = y_test.map(train_group_means) # Maps the training group means onto y_test categories
    X_test[col] = X_test[col].fillna(test_group_mean_series) # Fills NaNs using matching group average

# 6. Perform SMOTE analysis on TRAINING DATA ONLY
print("Group distribution before SMOTE:")
print(y_train.value_counts()) # Prints group count before SMOTE

smote = SMOTE(random_state=42) # Reproducible random seed (42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train) # Small samples for training set only

print("Group distribution after SMOTE:")
print(pd.Series(y_train_resampled).value_counts()) # Prints balanced group count after SMOTE

# 7. Data Frames for Model training
df_train_balanced = pd.DataFrame(X_train_resampled, columns=numeric_cols) # Adds numeric columns for resampled train set
df_train_balanced['diagnosis'] = y_train_resampled.values # Adds diagnosis labels
df_train_balanced.insert(0, 'participant_id', [f"train_sample_{i+1}" for i in range(len(df_train_balanced))]) # Adds unique participant_id column

# Reorder columns
front_cols = ['participant_id', 'diagnosis'] # Puts diagnosis and id far left
other_cols = [c for c in df_train_balanced.columns if c not in front_cols] # Remaining column names
df_train_balanced = df_train_balanced[front_cols + other_cols] # Orders training dataframe

# Test DataFrame
df_test_clean = pd.DataFrame(X_test, columns=numeric_cols) # Adds numeric columns for raw test set
df_test_clean['diagnosis'] = y_test.values # Adds real diagnosis labels
df_test_clean.insert(0, 'participant_id', [f"test_sample_{i+1}" for i in range(len(df_test_clean))]) # Adds unique participant_id column to far left

# Reorder columns
df_test_clean = df_test_clean[front_cols + other_cols] # Orders testing dataframe

# 8. Save both datasets to Drive
# File Path
output_dir = "/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Results/csv_files"

# Training CSV
df_train_balanced.to_csv(os.path.join(output_dir, "ppa_master_data_frame_train.csv"), index=False)
# Testing CSV
df_test_clean.to_csv(os.path.join(output_dir, "ppa_master_data_frame_test.csv"), index=False)

print(f"SUCCESS! Both CSV files saved! Ready for training the model!")

In [ ]:
# Check test set sample count per group
print(df_test_clean['diagnosis'].value_counts())

## Overview of all Master CSV Files Created

- **master_ppa_data_frame.csv**: Initial master CSV file that compiled all data from 12 CSV files
- **master_ppa_data_frame_updated.csv**: Added diagnoses (lvPPA, svPPA, nfvPPA, PPA, unknown, control)
- **ppa_master_data_frame_model.csv**: Removed participant row that lacked acoustic metrics, and had diagnosis of PPA (generic) or unknown
- **ppa_master_data_frame_train.csv**: 70% of SMOTE-applied data used for training ML model
- **ppa_master_data_frame_test.csv**: 30% of data (NO SMOTE) used for testing ML model

Saved in subfolder: **csv_files** in **Data** Folder

## Sample Size Table - Results
Sample Size table will include
- **Diagnosis (control, nfvPPA, lvPPA, svPPA)**
- **Total sample size (Pre-SMOTE train + No-SMOTE test)**
- **Pre-SMOTE training data sample size**
- **Post-SMOTE training data sample size**
- **No-SMOTE testing data sample size**
This table sets up essential context to explain data scarcity and class imbalance before diving into model's performance metrics


In [ ]:
# Import pandas to construct df
import pandas as pd

# Connect Drive and Colab
from google.colab import drive
drive.mount('/content/drive')

# Tell Pandas to show full column names with max column width without cutting them off
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

# Define the data table with your exact metrics
data = {
    'Diagnosis': [
        'control',
        'lvPPA',
        'nfvPPA',
        'svPPA',
        'Total Sample Size'
    ],
    'Class Sample Size': [244, 19, 10, 10, 283],
    'Training Set (Pre-SMOTE)': [171, 13, 7, 7, 198],
    'Training Set (Post-SMOTE)': [171, 171, 171, 171, 684],
    'Test Set (No SMOTE)': [73, 6, 3, 3, 85]
}

# Create the DataFrame
df_table = pd.DataFrame(data)

# Display the dataframe
display(df_table)

# Save CSV in drive after establishing file path
csv_file = "/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Results/figures_tables/Table_Dataset_Distribution_updated.csv"
df_table.to_csv(csv_file, index=False)